# Personal Finance Tracker: A Reproducible Data Science Demonstration

**Research questions:** Can merchant text classify spending better than a majority baseline? Can an unsupervised model identify injected unusual purchases? Does a lag-based forecast outperform last year's same-month expense?

This notebook is the main project demonstration. It generates five years of **synthetic USD transactions**, cleans CSVs, joins JSON receipts, builds a local document store and monthly marts, evaluates models chronologically, and exports reports. No external account, GPU, or dataset download is required.

Run with the installed Python environment or the local Docker Compose notebook service. All processing, storage, and reporting stay on your machine. All money is in one configured currency; results are educational, not financial advice. See `README.md` for setup and `ACTIVITIES.md` for the implementation walkthrough.

In [ ]:
%matplotlib inline

import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from finance_tracker.data import generate_demo
from finance_tracker.pipeline import run_pipeline
from finance_tracker.analytics import plot_overview

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / 'config' / 'demo.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from the cloned personal-finance-tracker folder')
CONFIG = json.loads((ROOT / 'config' / 'demo.json').read_text())
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
LOG = OUTPUT / 'notebook_activities.md'
LOG.write_text('# Notebook execution activities\n', encoding='utf-8')

def record_activity(title, explanation, result):
    timestamp = datetime.now(timezone.utc).isoformat()
    with LOG.open('a', encoding='utf-8') as stream:
        stream.write(f'\n## {title}\n\n{timestamp}\n\n{explanation}\n\n```text\n{result}\n```\n')

INPUT = ROOT / 'data' / 'demo'
generate_demo(INPUT, seed=CONFIG['seed'], years=CONFIG['years'])
results = run_pipeline(INPUT, OUTPUT, CONFIG['currency'], CONFIG['forecast_months'])
transactions = results['transactions']
monthly = results['monthly']
record_activity('Setup and processing',
                'Generated seeded synthetic data and ran the shared local Python pipeline.',
                json.dumps({'python': platform.python_version(), 'kernel': sys.executable,
                            'configuration': CONFIG, 'quality': results['metrics']['data_quality']}, indent=2))
display(Markdown(f'**Loaded {len(transactions):,} transactions across {len(monthly)} months.**'))
display(pd.Series(results['metrics']['data_quality'], name='Data-quality audit').to_frame())

## 1. Preprocessing and Exploratory Data Analysis

Dates are parsed to UTC; invalid dates, amounts, and blank IDs are rejected and counted. Duplicate transaction IDs are removed; conflicting dates/amounts and mixed currencies raise errors. Missing merchant names become `Unknown merchant`. Receipts enrich a bank transaction, rather than creating another expense. The raw generated CSV retains intentional quality problems for comparison.

Positive amounts represent income; negative amounts represent expenses. Transfer detection, refunds, exchange rates, and tax accounting need a real bank-specific adapter. Category labels below are recorded labels, not model predictions. Monthly cumulative net cash flow assumes zero opening balance and is not a statement balance.

In [ ]:
display(transactions.head())
display(transactions[['amount', 'expense']].describe())
display(transactions.groupby('category').agg(count=('transaction_id', 'size'), total_expense=('expense', 'sum')))
display(monthly.tail(12).round(2))
assert np.isclose(monthly.net_cash_flow.sum(), transactions.amount.sum())
figure = plot_overview(transactions, monthly, results['forecast'])
plt.show()

category_month = transactions.assign(month=transactions.date.dt.strftime('%Y-%m')).pivot_table(
    index='category', columns='month', values='expense', aggfunc='sum', fill_value=0)
figure, axis = plt.subplots(figsize=(15, 4), constrained_layout=True)
sns.heatmap(category_month, cmap='YlGnBu', ax=axis)
axis.set_title('Category spending over five years')
plt.show()
record_activity('EDA and reconciliation', 'Reviewed quality, distributions, cash flow, and category seasonality.',
                f'Cash flow reconciled: {monthly.net_cash_flow.sum():.2f} {CONFIG["currency"]}')

## 2. Supervised Categorization and Unsupervised Anomalies

The classifier uses TF-IDF merchant text plus standardized expense. Both preprocessing and model fitting use only the earlier 80% of labelled expenses; later transactions are the holdout. Macro F1 weights categories equally and is compared with a majority-class predictor. The generator repeats a small merchant vocabulary, so high scores are expected and do not prove generalization to new merchants. An actual study needs human-reviewed labels and a merchant-disjoint test.

Isolation Forest fits earlier log expenses and day-of-month without using injected-anomaly labels. Later synthetic labels allow a limited precision/recall demonstration. A flagged transaction is a review candidate, not proof of fraud. The fixed contamination rate is a modelling assumption, not an estimated fraud rate.

In [ ]:
classification = results['classification']
display(pd.Series(classification['metrics'], name='Classification').to_frame())
display(pd.DataFrame(classification['report']).T.round(3))
figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
sns.heatmap(classification['confusion'], annot=True, fmt='d', cmap='Greens',
            xticklabels=classification['labels'], yticklabels=classification['labels'], ax=axis)
axis.set(xlabel='Predicted', ylabel='Recorded category', title='Chronological holdout confusion matrix')
plt.show()

anomalies = results['anomalies']['transactions']
display(pd.Series(results['anomalies']['metrics'], name='Anomaly evaluation').to_frame())
display(anomalies.loc[anomalies.flagged, ['date', 'description', 'expense', 'anomaly_score', 'evaluation_holdout']].head(15))
figure, axis = plt.subplots(figsize=(12, 4), constrained_layout=True)
axis.scatter(anomalies.date, anomalies.expense, s=8, alpha=0.35, label='Expenses')
flagged = anomalies.loc[anomalies.flagged]
axis.scatter(flagged.date, flagged.expense, color='crimson', s=22, label='Flagged for review')
axis.set(title='Anomaly review timeline', ylabel=f'Expense ({CONFIG["currency"]})')
axis.legend()
plt.show()
record_activity('Model evaluation', 'Evaluated later transactions, with baselines and synthetic-label caveats.',
                json.dumps({'classification': classification['metrics'], 'anomalies': results['anomalies']['metrics']}, indent=2))

## 3. Forecasting and Budget Scenarios

Ridge regression uses previous-month expense, the previous three-month mean, last year's same month, and sine/cosine calendar features. A rolling one-step backtest over the final 12 months only trains on earlier data. Compare its MAE with the seasonal-naive baseline; lower is better and a more complex model need not win.

Future forecasts recurse on their own predictions. The shaded band in the overview uses the 90th percentile of absolute one-step backtest errors. It is not a calibrated 90% interval and does not capture growing multi-step uncertainty. Use at least 36 complete months; remove partial first/last months from real inputs. Missing calendar months become zero activity, so check export coverage first.

In [ ]:
forecast = results['forecast']
display(pd.Series(forecast['metrics'], name='Rolling backtest').to_frame())
display(forecast['future'].round(2))
figure, axis = plt.subplots(figsize=(12, 4), constrained_layout=True)
for column in ['actual', 'ridge', 'seasonal_baseline']:
    axis.plot(forecast['backtest'].index, forecast['backtest'][column], marker='o', label=column)
axis.set(title='Rolling one-month-ahead backtest', ylabel=f'Expense ({CONFIG["currency"]})')
axis.legend()
plt.show()
record_activity('Forecast comparison', 'Compared rolling Ridge MAE with seasonal baseline and exported future expenses.',
                json.dumps(forecast['metrics'], indent=2))

import ipywidgets as widgets

def spending_view(year, category):
    selected = transactions.loc[transactions.date.dt.year.eq(year)]
    if category != 'All':
        selected = selected.loc[selected.category.eq(category)]
    totals = selected.set_index('date').expense.resample('MS').sum()
    figure, axis = plt.subplots(figsize=(10, 3), constrained_layout=True)
    axis.plot(totals.index, totals.values, marker='o', color='teal')
    axis.set(title=f'{year}: {category}', ylabel=CONFIG['currency'])
    plt.show()

spending_controls = widgets.interactive(
    spending_view, year=widgets.Dropdown(options=sorted(transactions.date.dt.year.unique()), description='Year'),
    category=widgets.Dropdown(options=['All', *sorted(transactions.category.unique())], description='Category'))
display(spending_controls)

def budget_scenario(reduction_percent):
    scenario = forecast['future'][['predicted_expense']].copy()
    scenario['scenario_expense'] = scenario.predicted_expense * (1 - reduction_percent / 100)
    scenario['assumed_income'] = monthly.income.tail(3).mean()
    scenario['scenario_net_cash_flow'] = scenario.assumed_income - scenario.scenario_expense
    display(scenario.round(2))

display(widgets.interactive(budget_scenario, reduction_percent=widgets.IntSlider(
    value=10, min=0, max=40, step=5, description='Cut (%)', continuous_update=False)))

## 4. Local Document Storage and SQL Inspection

SQLite holds one JSON document per transaction, including any nested receipt. Repeated runs upsert by transaction ID. This is local document persistence, not a distributed NoSQL service. Pandas operates in memory and produces a monthly CSV mart; the SQL query below offers a second way to inspect stored records.

All inputs and reports live in the project's `data/` and `outputs/` directories, which persist on the host when using Docker Compose. No remote database, credentials, or external infrastructure is involved. Use a fresh output directory when changing datasets: upserts do not delete older transaction IDs that disappear from later input files.

In [ ]:
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(OUTPUT / 'transactions.sqlite3')) as connection:
    receipt_examples = pd.read_sql_query(
        "SELECT id, json_extract(payload, '$.receipt') AS receipt FROM transactions "
        "WHERE json_type(payload, '$.receipt') = 'object' LIMIT 3", connection)
    stored_monthly = pd.read_sql_query(
        "WITH ledger AS ("
        "SELECT substr(json_extract(payload, '$.date'), 1, 7) AS month, "
        "CAST(json_extract(payload, '$.amount') AS REAL) AS amount FROM transactions) "
        "SELECT month, COUNT(*) AS transactions, SUM(amount) AS net_cash_flow, "
        "SUM(CASE WHEN amount < 0 THEN -amount ELSE 0 END) AS expense "
        "FROM ledger GROUP BY month ORDER BY month", connection)

display(receipt_examples)
display(stored_monthly.tail(12).round(2))
print(f'Local document store: {OUTPUT / "transactions.sqlite3"}')
record_activity('Local document storage',
                'Inspected nested receipt JSON and queried monthly expenses/cash flow in SQLite; all processing stayed local.',
                stored_monthly.tail(12).round(2).to_string(index=False))

## 5. Findings, Reproducibility, and Limitations

Use the actual metrics below to discuss whether each model beats its baseline. Do not replace a worse result with an untested claim. The synthetic merchant vocabulary, anomaly injection process, and income/spending rules constrain what can be concluded.

For a stronger dissertation experiment: use consented anonymized records, retain a bank-specific data dictionary, compare merchant-disjoint classification, tune hyperparameters only on training-period folds, evaluate multiple forecast horizons, and assess drift. Pandas and SQLite do not prove distributed processing at enterprise scale. This project demonstrates reproducible local analytics rather than a big-data benchmark.

Reports are written under `outputs/` and are excluded from Git. The execution activity log records what happened in this run; `ACTIVITIES.md` explains implementation and setup. Run the same code through local Python or Docker Compose. Static HTML preserves charts and tables, but widget interaction needs a live notebook kernel.

In [ ]:
category_metrics = results['metrics']['classification']
forecast_metrics = results['metrics']['forecast']
summary = pd.DataFrame([
    {'experiment': 'Category macro F1 (higher is better)', 'model': category_metrics['macro_f1'],
     'baseline': category_metrics['majority_baseline_macro_f1']},
    {'experiment': 'Expense MAE (lower is better)', 'model': forecast_metrics['ridge_mae'],
     'baseline': forecast_metrics['seasonal_baseline_mae']},
])
display(summary.round(4))
record_activity('Conclusion', 'Saved observed model-versus-baseline results, not claims of real-world accuracy.', summary.to_string(index=False))
display(Markdown(f'**Reports:** `{OUTPUT}`\n\n**Execution log:** `{LOG}`'))
print(LOG.read_text(encoding='utf-8'))